# 03 · Hypothesis Testing

Five hypothesis tests against five distinct business questions:

| # | Test                    | Business question                                              |
|---|-------------------------|----------------------------------------------------------------|
| 1 | One-sample t-test       | Is the average first-innings total different from 160 runs?    |
| 2 | Welch two-sample t-test | Do first- and second-innings totals differ on average?         |
| 3 | Chi-square independence | Does the toss decision (bat/field) influence match outcome?    |
| 4 | One-way ANOVA           | Do venues differ in average first-innings score?               |
| 5 | One-proportion z-test   | Is the toss-winner win rate significantly above 50%?           |

In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_processed_or_build
from src.viz import savefig, PRIMARY, ACCENT, HIGHLIGHT

matches, deliveries, innings = load_processed_or_build()
ALPHA = 0.05
def verdict(p): return 'REJECT H0' if p < ALPHA else 'FAIL TO REJECT H0'
results = []

## 3.1 Test 1 — One-sample t-test on first-innings total

In [2]:
first_inn = innings[innings['inning'] == 1]
t, p = stats.ttest_1samp(first_inn['innings_total'], popmean=160)
print(f'H0: mean = 160 runs;  H1: mean != 160')
print(f't = {t:.4f}, df = {len(first_inn)-1}, p = {p:.4g}  -> {verdict(p)}')
results.append(['1. One-sample t (mu0=160)', float(t), float(p), verdict(p)])

H0: mean = 160 runs;  H1: mean != 160
t = 7.6236, df = 1192, p = 5.018e-14  -> REJECT H0


## 3.2 Test 2 — Welch's t-test: 1st vs 2nd innings totals

In [3]:
second_inn = innings[innings['inning'] == 2]
t, p = stats.ttest_ind(first_inn['innings_total'], second_inn['innings_total'], equal_var=False)
print(f'Mean 1st inn = {first_inn["innings_total"].mean():.2f}')
print(f'Mean 2nd inn = {second_inn["innings_total"].mean():.2f}')
print(f't = {t:.4f}, p = {p:.4g}  -> {verdict(p)}')
results.append(['2. Welch t (1st vs 2nd)', float(t), float(p), verdict(p)])

Mean 1st inn = 167.36
Mean 2nd inn = 154.07
t = 9.8087, p = 2.719e-22  -> REJECT H0


## 3.3 Test 3 — Chi-square: toss decision vs match outcome

In [4]:
m = matches.dropna(subset=['winner']).copy()
m['toss_winner_won'] = (m['toss_winner'] == m['winner']).astype(int)
ct = pd.crosstab(m['toss_decision'], m['toss_winner_won'])
ct.columns = ['Toss-winner LOST', 'Toss-winner WON']
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(ct)
print(f'\nchi2 = {chi2:.4f}, dof = {dof}, p = {p:.4g}  -> {verdict(p)}')
results.append(['3. Chi-square (decision x outcome)', float(chi2), float(p), verdict(p)])

               Toss-winner LOST  Toss-winner WON
toss_decision                                   
bat                         219              184
field                       359              422

chi2 = 7.1322, dof = 1, p = 0.007571  -> REJECT H0


## 3.4 Test 4 — One-way ANOVA across top-6 venues

In [5]:
top_v = first_inn['venue'].value_counts().head(6).index.tolist()
groups = [first_inn.loc[first_inn['venue'] == v, 'innings_total'].values for v in top_v]
F, p = stats.f_oneway(*groups)
for v, g in zip(top_v, groups):
    print(f'  {v[:45]:46s} n={len(g):4d}  mean={np.mean(g):.2f}')
print(f'\nF = {F:.4f}, p = {p:.4g}  -> {verdict(p)}')
results.append(['4. ANOVA (top-6 venues)', float(F), float(p), verdict(p)])

  Wankhede Stadium                               n= 128  mean=171.64
  Eden Gardens                                   n= 103  mean=167.88
  M Chinnaswamy Stadium                          n= 102  mean=173.75
  Arun Jaitley Stadium                           n=  99  mean=172.06
  MA Chidambaram Stadium                         n=  94  mean=165.16
  Rajiv Gandhi International Stadium             n=  85  mean=165.41

F = 1.0662, p = 0.378  -> FAIL TO REJECT H0


## 3.5 Test 5 — One-proportion z-test: toss winners > 50%?

In [6]:
n_m, x = len(m), int(m['toss_winner_won'].sum())
p_hat = x / n_m
se = np.sqrt(0.5 * 0.5 / n_m)
z = (p_hat - 0.5) / se
p = 1 - stats.norm.cdf(z)  # one-sided
print(f'p_hat = {p_hat:.4f}  (x={x}, n={n_m})')
print(f'z = {z:.4f}, one-sided p = {p:.4g}  -> {verdict(p)}')
results.append(['5. Z-test (toss-win > 0.5)', float(z), float(p), verdict(p)])

p_hat = 0.5118  (x=606, n=1184)
z = 0.8137, one-sided p = 0.2079  -> FAIL TO REJECT H0


## 3.6 Compact summary table

In [7]:
summary = pd.DataFrame(results, columns=['Test', 'Statistic', 'p-value', 'Decision'])
summary['Statistic'] = summary['Statistic'].round(4)
summary['p-value'] = summary['p-value'].apply(lambda v: f'{v:.2e}' if v < 0.001 else f'{v:.4f}')
summary.to_csv('reports/tables/hypothesis_tests.csv', index=False)
summary

,Test,Statistic,p-value,Decision
0,1. One-sample t (mu0=160),7.6236,5.02e-14,REJECT H0
1,2. Welch t (1st vs 2nd),9.8087,2.72e-22,REJECT H0
2,3. Chi-square (decision x outcome),7.1322,0.0076,REJECT H0
3,4. ANOVA (top-6 venues),1.0662,0.3780,FAIL TO REJECT H0
4,5. Z-test (toss-win > 0.5),0.8137,0.2079,FAIL TO REJECT H0


In [8]:
# Decision visualisation
fig, ax = plt.subplots(figsize=(10, 4.5))
raw_p = [r[2] for r in results]
names = [r[0].split('.')[1].strip() for r in results]
colors = [ACCENT if pv < ALPHA else PRIMARY for pv in raw_p]
log_p = [-np.log10(max(pv, 1e-12)) for pv in raw_p]
bars = ax.barh(names, log_p, color=colors)
ax.axvline(-np.log10(ALPHA), color='gray', linestyle='--',
           label=f'-log10(alpha = {ALPHA}) = {-np.log10(ALPHA):.2f}')
ax.set_xlabel('-log10(p-value)')
ax.set_title('Hypothesis Tests: Strength of Evidence Against H0')
ax.legend()
for b, pv in zip(bars, raw_p):
    label = f'p={pv:.2e}' if pv < 0.001 else f'p={pv:.3f}'
    ax.text(b.get_width() + 0.05, b.get_y() + b.get_height()/2,
            label, va='center', fontsize=10)
savefig('hyp_01_pvalue_summary.png')
plt.show()

## 3.7 Interpretation

* Where we **reject H₀**, we have statistical evidence at the 5% level that the effect exists. Quote the *effect size* alongside the p-value — a tiny effect can be statistically significant on a large dataset and still be managerially irrelevant.
* Where we **fail to reject H₀**, that is not the same as proving H₀: it means our data is consistent with the null.